In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("india_2000_2024_daily_weather.csv")

# choose one city (change if needed)
df = df[df["city"] == "Mumbai"]

df["date"] = pd.to_datetime(df["date"], dayfirst=True)
df = df.sort_values("date")

def get_state(row):
    
    temp = row["temperature_2m_max"]
    rain = row["rain_sum"]
    wind = row["wind_speed_10m_max"]
    
    if wind >= 40:
        return "Storm"
    
    if temp >= 36:
        return "Heatwave"
    
    if rain >= 20:
        return "Heavy Rain"
    
    if rain > 0:
        return "Rainy"
    
    if 25 <= temp <= 35:
        return "Sunny"
    
    return "Cloudy"


df["state"] = df.apply(get_state, axis=1)


old = df[df["date"] < "2008"]
mid = df[(df["date"] >= "2008") & (df["date"] < "2016")]
new = df[df["date"] >= "2016"]


def transition_matrix(data):

    df = data.copy()
    df["next_state"] = df["state"].shift(-1)

    t = pd.crosstab(
        df["state"],
        df["next_state"],
        normalize="index"
    )

    return t

T_old = transition_matrix(old)
T_mid = transition_matrix(mid)
T_new = transition_matrix(new)

print("\nOLD MATRIX\n",T_old)
print("\nMID MATRIX\n",T_mid)
print("\nNEW MATRIX\n",T_new)


T_adaptive = (T_old.fillna(0) + T_mid.fillna(0) + T_new.fillna(0)) / 3

print("\nADAPTIVE MATRIX\n",T_adaptive)


states = T_adaptive.index.tolist()
P = T_adaptive.values

def predict(start, days):
    
    prob = np.zeros(len(states))
    prob[states.index(start)] = 1
    
    for _ in range(days):
        prob = prob @ P
        
    return dict(zip(states, prob))


print("\nPrediction after 5 days")
print(predict("Rainy",5))


def extreme_prob(result):
    
    extreme = ["Heavy Rain","Heatwave","Storm"]
    
    p = 0
    for e in extreme:
        if e in result:
            p += result[e]
            
    return p

res = predict("Sunny",3)

print("\nExtreme probability")
print(extreme_prob(res))


def simulate(start, days):
    
    current = start
    sim = [current]
    
    for _ in range(days):
        probs = T_adaptive.loc[current].values
        current = np.random.choice(states, p=probs)
        sim.append(current)
        
    return sim

simulation = simulate("Sunny",30)

print("\nSimulation")
print(simulation)


plt.figure(figsize=(10,6))
sns.heatmap(T_adaptive, annot=True, cmap="coolwarm")
plt.title("Adaptive Markov Chain Transition Matrix")
plt.show()

plt.figure(figsize=(12,4))
plt.plot(simulation)
plt.title("Weather Simulation")
plt.show()

: 

In [13]:
def markov_accuracy(df, matrix):

    states = matrix.index.tolist()
    P = matrix.values

    correct = 0
    total = 0

    df = df.copy()
    df["next_state"] = df["state"].shift(-1)

    for i in range(len(df)-1):

        current = df.iloc[i]["state"]
        actual = df.iloc[i]["next_state"]

        if current not in states:
            continue

        probs = matrix.loc[current].values
        predicted = states[np.argmax(probs)]

        if predicted == actual:
            correct += 1

        total += 1

    return correct / total

In [14]:
accuracy = markov_accuracy(df, T_adaptive)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.8576278611324061
